# Network Analysis: Combined 2017-2019

This notebook analyzes genre collaboration networks across all three years (2017-2019):
- Network structure and properties
- Degree distributions
- Edge weight and popularity distributions
- Top collaborating genres
- Year-over-year comparison

## Import Libraries

In [1]:
import os
os.chdir('..') 
print("Now working from:", os.getcwd())

Now working from: /Users/mikolajandrzejewski/Documents/GitHub


In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Loading Data for All Years

In [ ]:
years = [2017, 2018, 2019]
networks = {}
dataframes = {}

for year in years:
    print(f"\nLoading {year} data...")
    df = pd.read_csv(
        f"../dataset/Original/global/global-genre_network-{year}.csv",
        sep="\t"
    )
    dataframes[year] = df
    print(f"  {len(df)} collaboration pairs")
    print(f"  Columns: {list(df.columns)}")

print("\nData loaded for all years!")

**Column descriptions:**
- `source` = first genre in collaboration
- `target` = second genre in collaboration  
- `weight` = number of times these genres collaborated (co-appeared in charts)
- `avg_streams` = average streams for songs with this genre pair

## Data Validation

In [ ]:
for year in years:
    df = dataframes[year]
    print(f"\n{'='*50}")
    print(f"Year {year}")
    print(f"{'='*50}")
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nBasic statistics:")
    print(df.describe())

## Collaboration Network Construction

In [ ]:
# Build networks for all years
for year in years:
    df = dataframes[year]
    G = nx.from_pandas_edgelist(
        df,
        source='source',
        target='target',
        edge_attr=['weight', 'avg_streams'],
        create_using=nx.Graph()
    )
    networks[year] = G
    print(f"{year}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Network Overview - All Years

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, year in enumerate(years):
    G = networks[year]
    ax = axes[idx]
    
    # Draw network
    pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=20, node_color='skyblue', alpha=0.6)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2, width=0.5)
    
    ax.set_title(f'{year} Genre Network', fontsize=14, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('network analysis/network_overview_all_years.png', dpi=200, bbox_inches='tight')
plt.show()

print("Network visualizations saved!")

## Network Metrics Comparison

In [ ]:
metrics = []

for year in years:
    G = networks[year]
    
    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    density = nx.density(G)
    
    metrics.append({
        'Year': year,
        'Nodes': G.number_of_nodes(),
        'Edges': G.number_of_edges(),
        'Avg Degree': round(avg_degree, 2),
        'Density': round(density, 4)
    })

df_metrics = pd.DataFrame(metrics)
print("\nNetwork Metrics Comparison:")
print(df_metrics.to_string(index=False))

## Node Degree Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, year in enumerate(years):
    G = networks[year]
    degrees = [deg for _, deg in G.degree()]
    
    ax = axes[idx]
    ax.hist(degrees, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Degree (number of collaborations)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{year} - Degree Distribution', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('network analysis/degree_distribution_all_years.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Degree statistics
for year in years:
    G = networks[year]
    degrees = [deg for _, deg in G.degree()]
    print(f"\n{year} Degree Statistics:")
    print(f"  Mean: {np.mean(degrees):.2f}")
    print(f"  Median: {np.median(degrees):.2f}")
    print(f"  Max: {max(degrees)}")
    print(f"  Min: {min(degrees)}")

## Edge Weight Distribution

Weight = number of times two genres collaborated (co-appeared on charts)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, year in enumerate(years):
    G = networks[year]
    weights = [d["weight"] for _, _, d in G.edges(data=True)]
    
    ax = axes[idx]
    ax.hist(weights, bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Weight (collaboration frequency)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{year} - Weight Distribution', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('network analysis/weight_distribution_all_years.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nObservation: Most genre pairs occur very few times (1–3)")
print("A small number of pairs occur frequently (heavy tail)")

In [ ]:
# Weight statistics
for year in years:
    G = networks[year]
    weights = [d["weight"] for _, _, d in G.edges(data=True)]
    print(f"\n{year} Weight Statistics:")
    print(f"  Mean: {np.mean(weights):.2f}")
    print(f"  Median: {np.median(weights):.2f}")
    print(f"  Max: {max(weights)}")
    print(f"  Pairs with weight=1: {sum(1 for w in weights if w == 1)} ({100*sum(1 for w in weights if w == 1)/len(weights):.1f}%)")

## Popularity Distribution (avg_streams)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, year in enumerate(years):
    G = networks[year]
    streams = [d["avg_streams"] for _, _, d in G.edges(data=True)]
    
    ax = axes[idx]
    ax.hist(streams, bins=40, color='lightblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Average Streams', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{year} - Popularity Distribution', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('network analysis/streams_distribution_all_years.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nNote: Even the 'least successful' collaborations have millions of streams")
print("(Selection bias: only chart-hitting songs are in the dataset)")

## Top 10 Most Connected Genres (by degree)

In [ ]:
for year in years:
    G = networks[year]
    top_genres = sorted(G.degree, key=lambda x: x[1], reverse=True)[:10]
    
    print(f"\n{'='*60}")
    print(f"{year} - Top 10 Genres by Collaboration Count")
    print(f"{'='*60}")
    for rank, (genre, degree) in enumerate(top_genres, 1):
        print(f"{rank:2d}. {genre:30s} - {degree:3d} collaborations")

## Top 10 Most Frequent Collaborations (by weight)

In [ ]:
for year in years:
    G = networks[year]
    top_edges = sorted(G.edges(data=True), key=lambda x: x[2]["weight"], reverse=True)[:10]
    
    print(f"\n{'='*70}")
    print(f"{year} - Top 10 Most Frequent Genre Pairs")
    print(f"{'='*70}")
    for rank, (source, target, data) in enumerate(top_edges, 1):
        print(f"{rank:2d}. {source:25s} ↔ {target:25s} - {data['weight']:3d} times")

## Top 10 Most Successful Collaborations (by avg_streams)

In [ ]:
for year in years:
    G = networks[year]
    top_edges = sorted(G.edges(data=True), key=lambda x: x[2]["avg_streams"], reverse=True)[:10]
    
    print(f"\n{'='*80}")
    print(f"{year} - Top 10 Most Successful Collaborations (by streams)")
    print(f"{'='*80}")
    for rank, (source, target, data) in enumerate(top_edges, 1):
        streams_millions = data['avg_streams'] / 1_000_000
        print(f"{rank:2d}. {source:25s} ↔ {target:25s} - {streams_millions:6.1f}M streams")

## Key Observation

**Collaboration frequency ≠ popularity**

The most frequently collaborating genre pairs are not necessarily the most successful in terms of streams. This suggests that:
- Some niche collaborations can be extremely popular
- Frequent collaboration doesn't guarantee high engagement
- Quality/novelty may matter more than quantity

## Year-over-Year Trends

In [ ]:
# Compare metrics across years
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Nodes over time
axes[0, 0].plot(years, [networks[y].number_of_nodes() for y in years], marker='o', linewidth=2, markersize=8)
axes[0, 0].set_title('Number of Genres', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(alpha=0.3)

# Edges over time
axes[0, 1].plot(years, [networks[y].number_of_edges() for y in years], marker='o', linewidth=2, markersize=8, color='orange')
axes[0, 1].set_title('Number of Collaborations', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(alpha=0.3)

# Average degree over time
avg_degrees = [sum(dict(networks[y].degree()).values()) / networks[y].number_of_nodes() for y in years]
axes[1, 0].plot(years, avg_degrees, marker='o', linewidth=2, markersize=8, color='green')
axes[1, 0].set_title('Average Degree', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Avg Degree')
axes[1, 0].grid(alpha=0.3)

# Density over time
densities = [nx.density(networks[y]) for y in years]
axes[1, 1].plot(years, densities, marker='o', linewidth=2, markersize=8, color='red')
axes[1, 1].set_title('Network Density', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Density')
axes[1, 1].grid(alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Year')
    ax.set_xticks(years)

plt.tight_layout()
plt.savefig('network analysis/year_over_year_trends.png', dpi=200, bbox_inches='tight')
plt.show()

## Summary Statistics Table

In [ ]:
summary = []

for year in years:
    G = networks[year]
    df = dataframes[year]
    
    degrees = [deg for _, deg in G.degree()]
    weights = [d["weight"] for _, _, d in G.edges(data=True)]
    streams = [d["avg_streams"] for _, _, d in G.edges(data=True)]
    
    summary.append({
        'Year': year,
        'Genres': G.number_of_nodes(),
        'Collaborations': G.number_of_edges(),
        'Avg Degree': round(np.mean(degrees), 2),
        'Avg Weight': round(np.mean(weights), 2),
        'Avg Streams (M)': round(np.mean(streams) / 1_000_000, 2),
        'Density': round(nx.density(G), 4)
    })

df_summary = pd.DataFrame(summary)
print("\n" + "="*80)
print("NETWORK ANALYSIS SUMMARY (2017-2019)")
print("="*80)
print(df_summary.to_string(index=False))

# Save to CSV
df_summary.to_csv('network analysis/network_summary_2017-2019.csv', index=False)
print("\nSummary saved to 'network analysis/network_summary_2017-2019.csv'")